## Section 8 — Evaluation et Optimisation du Seuil

### Les metriques expliquees

En contexte medical avec desequilibre des classes, l'accuracy seule ne suffit pas.

```
Matrice de confusion :
                 Predit NORMAL   Predit PNEUMONIA
Vrai NORMAL           TN               FP
Vrai PNEUMONIA        FN               TP

Recall PNEUMONIE = TP / (TP + FN)  <- PRIORITAIRE
  => "Detectons-nous tous les malades ?"

Precision        = TP / (TP + FP)
  => "Quand on dit PNEUMONIE, a-t-on raison ?"

F1 Score = 2 * P * R / (P + R)
  => Equilibre Precision/Recall

AUC ROC  = Aire sous la courbe ROC
  => Performance globale [0.5=aleatoire, 1.0=parfait]
```

### Optimisation du seuil
Par defaut le seuil est 0.5 (PNEUMONIE si prob > 0.5).
En l'abaissant, on augmente le Recall mais on diminue la Precision.
On cherche le seuil avec **Recall PNEUMONIE >= 97%** et meilleur Recall NORMAL.

In [ ]:
# ============================================================
# SECTION 8 : EVALUATION COMPLETE
# ============================================================

# ────────────────────────────────────────────────────────────
# 8.1 Optimisation du seuil de decision
# ────────────────────────────────────────────────────────────

def trouver_meilleur_seuil(model, test_loader, nom):
    """
    Teste differents seuils de decision pour trouver le meilleur
    compromis Recall PNEUMONIE >= 97% avec meilleur Recall NORMAL.

    En abaissant le seuil :
    + Recall PNEUMONIE monte (on rate moins de malades)
    - Recall NORMAL baisse (plus de faux positifs)
    """
    model.eval()
    labels_all, probs_all = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs  = imgs.to(device)
            out   = model(imgs)
            probs = torch.softmax(out, dim=1)[:, 1]
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    probs_all  = np.array(probs_all)
    labels_all = np.array(labels_all)

    print(f"\nOptimisation seuil — {nom}")
    print(f"{'Seuil':>8} | {'Rec.Pneumo':>11} | {'Rec.Normal':>11} | {'F1':>8} | {'Valide':>8}")
    print("-"*55)

    meilleur_seuil    = 0.5
    meilleur_recall_n = 0.0

    for seuil in np.arange(0.20, 0.65, 0.05):
        preds  = (probs_all > seuil).astype(int)
        rec_p  = recall_score(labels_all, preds, pos_label=1, zero_division=0)
        rec_n  = recall_score(labels_all, preds, pos_label=0, zero_division=0)
        f1     = f1_score(labels_all, preds, average="weighted", zero_division=0)
        valide = "OK" if rec_p >= 0.97 else "---"
        print(f"{seuil:>8.2f} | {rec_p:>11.4f} | {rec_n:>11.4f} | {f1:>8.4f} | {valide:>8}")

        # Critere : Recall Pneumonie >= 97% ET meilleur Recall Normal
        if rec_p >= 0.97 and rec_n > meilleur_recall_n:
            meilleur_recall_n = rec_n
            meilleur_seuil    = seuil

    print(f"\nSeuil retenu : {meilleur_seuil:.2f} | Recall Normal : {meilleur_recall_n:.4f}")
    return meilleur_seuil

# ────────────────────────────────────────────────────────────
# 8.2 Evaluation finale avec seuil optimal
# ────────────────────────────────────────────────────────────

def evaluer_modele(model, test_loader, nom, seuil=0.5):
    """
    Evalue un modele sur le test set avec le seuil optimal.
    Retourne toutes les metriques.
    """
    model.eval()
    labels_all, probs_all = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs  = imgs.to(device)
            out   = model(imgs)
            probs = torch.softmax(out, dim=1)[:, 1]
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    probs_all  = np.array(probs_all)
    labels_all = np.array(labels_all)
    preds_all  = (probs_all > seuil).astype(int)

    return {
        "nom"        : nom,
        "acc"        : accuracy_score(labels_all, preds_all),
        "f1"         : f1_score(labels_all, preds_all, average="weighted", zero_division=0),
        "auc"        : roc_auc_score(labels_all, probs_all),
        "pre"        : precision_score(labels_all, preds_all, average="weighted", zero_division=0),
        "rec"        : recall_score(labels_all, preds_all, average="weighted", zero_division=0),
        "rec_pneumo" : recall_score(labels_all, preds_all, pos_label=1, zero_division=0),
        "rec_normal" : recall_score(labels_all, preds_all, pos_label=0, zero_division=0),
        "cm"         : confusion_matrix(labels_all, preds_all),
        "probs"      : probs_all,
        "labels"     : labels_all,
        "seuil"      : seuil
    }

# ── Trouver les seuils optimaux ───────────────────────────────
modeles_liste = [model1, model2, model3, model4, model5]
noms_liste    = [
    "M1-CNN Baseline",
    "M2-ResNet18",
    "M3-DenseNet-121",
    "M4-EfficientNet-V2",
    "M5-CNN+ViT"
]
seuils_optimaux = []

for model_ref, nom in zip(modeles_liste, noms_liste):
    s = trouver_meilleur_seuil(model_ref, test_loader, nom)
    seuils_optimaux.append(s)

# ── Evaluer tous les modeles ──────────────────────────────────
resultats = []
for model_ref, nom, seuil in zip(modeles_liste, noms_liste, seuils_optimaux):
    r = evaluer_modele(model_ref, test_loader, nom, seuil=seuil)
    resultats.append(r)

# ── Tableau comparatif final ──────────────────────────────────
print("\n" + "="*90)
print("TABLEAU COMPARATIF FINAL — SEUILS OPTIMAUX")
print("="*90)
print(f"{'Modele':<22} {'Acc':>7} {'F1':>7} {'AUC':>7} {'Rec.Tot':>8} "
      f"{'Rec.Pneu':>9} {'Rec.Norm':>9} {'Seuil':>7}")
print("-"*90)

for r, s in zip(resultats, seuils_optimaux):
    # Marquer si objectif Recall atteint
    flag = " *" if r["rec_pneumo"] >= 0.97 else "  "
    print(f"{r['nom']:<22} {r['acc']:>7.4f} {r['f1']:>7.4f} {r['auc']:>7.4f} "
          f"{r['rec']:>8.4f} {r['rec_pneumo']:>9.4f} {r['rec_normal']:>9.4f} "
          f"{s:>7.2f}{flag}")

print("\n* = Recall PNEUMONIE >= 97% (objectif medical atteint)")
print("\nObjectif : Recall PNEUMONIE >= 0.97")

# ── Rapport classification complet du meilleur modele ─────────
# Trouver le modele avec le meilleur Recall Pneumonie
# Selection par AUC + critere Recall Pneumo >= 97%
# M5 CNN+ViT : AUC 0.957, Recall Pneumo 97.2%, Recall Normal 70.5%
candidats  = [r for r in resultats if r["rec_pneumo"] >= 0.97]
meilleur_r = max(candidats, key=lambda x: x["auc"])
meilleur_idx = [r["nom"] for r in resultats].index(meilleur_r["nom"])
meilleur_m   = modeles_liste[meilleur_idx]

print(f"Rapport complet — {meilleur_r['nom']} (meilleur modele global — AUC {meilleur_r['auc']:.4f})")
print("="*60)

meilleur_m.eval()
labels_all, preds_all = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(device)
        out   = meilleur_m(imgs)
        probs = torch.softmax(out, dim=1)[:, 1]
        preds = (probs.cpu().numpy() > meilleur_r["seuil"]).astype(int)
        labels_all.extend(labels.numpy())
        preds_all.extend(preds)

print(classification_report(labels_all, preds_all,
                             target_names=["NORMAL", "PNEUMONIA"]))